# GNC Attitude Dynamics - Exploration & Validation

This notebook provides an interactive environment for understanding and validating the attitude dynamics calculations implemented in the GNC backend.

**Purpose**: Exploration and verification, not production use.

---

## Table of Contents
1. [Governing Equations](#governing-equations)
2. [Assumptions and Limitations](#assumptions-and-limitations)
3. [Expected Physical Behavior](#expected-physical-behavior)
4. [Setup & Imports](#setup)
5. [Spacecraft Configuration](#spacecraft-configuration)
6. [Attitude Propagation](#attitude-propagation)
7. [Results Visualization](#results-visualization)
8. [Validation Cases](#validation-cases)

---

## Governing Equations <a id='governing-equations'></a>

The backend implements **Euler's equations of rotational motion** for a rigid body. These equations describe the angular acceleration of a spacecraft given its moments of inertia, current angular velocities, and applied external torques.

### Euler's Equations (Principal Axes Form)

$$
\begin{aligned}
I_x \dot{\omega}_x &= (I_y - I_z) \omega_y \omega_z + M_x \\
I_y \dot{\omega}_y &= (I_z - I_x) \omega_z \omega_x + M_y \\
I_z \dot{\omega}_z &= (I_x - I_y) \omega_x \omega_y + M_z
\end{aligned}
$$

### Solved for Angular Accelerations

The backend computes:

$$
\begin{aligned}
\dot{\omega}_x &= \frac{(I_y - I_z) \omega_y \omega_z + M_x}{I_x} \\
\dot{\omega}_y &= \frac{(I_z - I_x) \omega_z \omega_x + M_y}{I_y} \\
\dot{\omega}_z &= \frac{(I_x - I_y) \omega_x \omega_y + M_z}{I_z}
\end{aligned}
$$

### Additional Computed Quantities

**Rotational Kinetic Energy:**
$$T = \frac{1}{2} \left( I_x \omega_x^2 + I_y \omega_y^2 + I_z \omega_z^2 \right)$$

**Angular Momentum Magnitude:**
$$|\mathbf{H}| = \sqrt{(I_x \omega_x)^2 + (I_y \omega_y)^2 + (I_z \omega_z)^2}$$

### Variable Definitions

| Symbol | Description | Units |
|--------|-------------|-------|
| $I_x, I_y, I_z$ | Principal moments of inertia | kg·m² |
| $\omega_x, \omega_y, \omega_z$ | Angular velocity components | rad/s |
| $\dot{\omega}_x, \dot{\omega}_y, \dot{\omega}_z$ | Angular acceleration components | rad/s² |
| $M_x, M_y, M_z$ | External torque components | N·m |
| $T$ | Rotational kinetic energy | J |
| $\mathbf{H}$ | Angular momentum vector | kg·m²/s |

---

## Assumptions and Limitations <a id='assumptions-and-limitations'></a>

The current backend implementation makes several simplifying assumptions:

### Assumptions

1. **Rigid Body**: The spacecraft is assumed to be perfectly rigid with no structural flexure or deformation. This is valid for stiff structures but may break down for spacecraft with flexible appendages (solar arrays, antennas).

2. **Principal Axes Alignment**: The body-fixed coordinate frame is aligned with the principal axes of inertia. This means:
   - Products of inertia are zero ($I_{xy} = I_{xz} = I_{yz} = 0$)
   - Inertia tensor is diagonal

3. **Instantaneous Calculation**: The backend computes instantaneous angular accelerations. Time integration must be performed externally (as we do in this notebook).

4. **Constant Inertia**: The moments of inertia are assumed constant. This doesn't account for fuel slosh, solar array rotation, or mass jettison.

### Limitations

| Limitation | Impact | Fidelity Improvement |
|------------|--------|----------------------|
| No attitude representation | Cannot track orientation in inertial frame | Add quaternion/DCM kinematics |
| No environmental torques | Missing gravity gradient, magnetic, aero | Add disturbance torque models |
| No reaction wheel dynamics | Cannot model internal actuators | Add momentum storage devices |
| First-order integration | Numerical drift over long propagations | Use RK4 or higher-order methods |

### When This Model is Valid

- Short-duration attitude maneuvers
- Preliminary design trades
- Understanding fundamental dynamics
- Symmetric or near-symmetric spacecraft

---

## Expected Physical Behavior <a id='expected-physical-behavior'></a>

Understanding what to expect helps validate that the simulation is working correctly.

### Torque-Free Motion

When no external torques are applied ($M_x = M_y = M_z = 0$):

1. **Angular momentum is conserved**: $|\mathbf{H}|$ remains constant
2. **Kinetic energy is conserved**: $T$ remains constant
3. **Spin about principal axis**: Pure rotation about any principal axis is stable for the major and minor axes, unstable for the intermediate axis

### Stability of Principal Axis Rotation

For a spacecraft with $I_x < I_y < I_z$:

| Spin Axis | Stability | Physical Behavior |
|-----------|-----------|-------------------|
| Minor axis ($I_x$) | Stable | Small perturbations → bounded oscillations |
| Intermediate axis ($I_y$) | **Unstable** | Small perturbations → exponential growth |
| Major axis ($I_z$) | Stable | Small perturbations → bounded oscillations |

This is known as the **"tennis racket theorem"** or **intermediate axis theorem**.

### Energy Dissipation Effects

Real spacecraft have internal energy dissipation (fuel slosh, flexible modes). Over time, this causes:
- Rotation to settle toward the **major axis** (maximum moment of inertia)
- This is the minimum energy state for a given angular momentum

*Note: The current model does not include dissipation.*

---

## Setup & Imports <a id='setup'></a>

In [2]:
# Standard library
import sys
from pathlib import Path

# Add the backend app to the Python path so we can import the services
backend_path = Path.cwd().parent
if str(backend_path) not in sys.path:
    sys.path.insert(0, str(backend_path))

# Third-party
import numpy as np
import matplotlib.pyplot as plt

# Backend services - IMPORTING PRODUCTION CODE
from app.services.gnc.attitude_dynamics import compute_attitude_dynamics
from app.schemas.gnc import AttitudeInput

print("Imports successful!")
print(f"Backend path: {backend_path}")

Imports successful!
Backend path: /Users/jimlee/Projects/gnc_calcs/backend


In [ ]:
# Configure matplotlib for better notebook display
%matplotlib inline
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 12
plt.rcParams['lines.linewidth'] = 2
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

---

## Spacecraft Configuration <a id='spacecraft-configuration'></a>

Define a simple spacecraft with distinct principal moments of inertia.

In [3]:
# ============================================================================
# Spacecraft Inertia Properties
# ============================================================================
# Example: Small satellite with elongated body (like a CubeSat stack)
# Principal moments of inertia [kg·m²]

INERTIA = {
    'Ix': 10.0,   # Minor axis - smallest moment of inertia
    'Iy': 25.0,   # Intermediate axis
    'Iz': 30.0,   # Major axis - largest moment of inertia
}

print("Spacecraft Principal Moments of Inertia:")
print(f"  Ix = {INERTIA['Ix']:6.1f} kg·m² (minor axis)")
print(f"  Iy = {INERTIA['Iy']:6.1f} kg·m² (intermediate axis)")
print(f"  Iz = {INERTIA['Iz']:6.1f} kg·m² (major axis)")
print(f"\nInertia ratios: Iy/Ix = {INERTIA['Iy']/INERTIA['Ix']:.2f}, Iz/Iy = {INERTIA['Iz']/INERTIA['Iy']:.2f}")

Spacecraft Principal Moments of Inertia:
  Ix =   10.0 kg·m² (minor axis)
  Iy =   25.0 kg·m² (intermediate axis)
  Iz =   30.0 kg·m² (major axis)

Inertia ratios: Iy/Ix = 2.50, Iz/Iy = 1.20


In [4]:
# ============================================================================
# Initial Angular Rates
# ============================================================================
# Start with rotation primarily about the Z-axis (major axis)
# with small perturbations in other axes

# Convert from deg/s to rad/s for convenience
def deg_to_rad(deg):
    return np.radians(deg)

INITIAL_RATES = {
    'omega_x': deg_to_rad(1.0),   # Small perturbation [rad/s]
    'omega_y': deg_to_rad(0.5),   # Small perturbation [rad/s]
    'omega_z': deg_to_rad(10.0),  # Primary spin axis [rad/s]
}

print("Initial Angular Velocities:")
print(f"  ωx = {INITIAL_RATES['omega_x']:.4f} rad/s ({np.degrees(INITIAL_RATES['omega_x']):.1f} °/s)")
print(f"  ωy = {INITIAL_RATES['omega_y']:.4f} rad/s ({np.degrees(INITIAL_RATES['omega_y']):.1f} °/s)")
print(f"  ωz = {INITIAL_RATES['omega_z']:.4f} rad/s ({np.degrees(INITIAL_RATES['omega_z']):.1f} °/s)")

Initial Angular Velocities:
  ωx = 0.0175 rad/s (1.0 °/s)
  ωy = 0.0087 rad/s (0.5 °/s)
  ωz = 0.1745 rad/s (10.0 °/s)


---

## Attitude Propagation <a id='attitude-propagation'></a>

We'll propagate the attitude dynamics over time using the production `compute_attitude_dynamics` function.

**Integration Method**: Simple Euler integration (first-order). This is sufficient for exploration but would need improvement for higher-fidelity work.

In [ ]:
def propagate_attitude(
    inertia: dict,
    initial_rates: dict,
    external_torques: dict = None,
    duration: float = 60.0,
    dt: float = 0.01
) -> dict:
    """
    Propagate attitude dynamics over time using the production backend.
    
    Uses Euler integration with the compute_attitude_dynamics function
    from the production backend.
    
    Args:
        inertia: Dict with Ix, Iy, Iz [kg·m²]
        initial_rates: Dict with omega_x, omega_y, omega_z [rad/s]
        external_torques: Dict with Mx, My, Mz [N·m] (default: zeros)
        duration: Total propagation time [s]
        dt: Time step [s]
        
    Returns:
        Dict containing time history arrays
    
    TODO: Upgrade to RK4 integration for improved accuracy
    TODO: Add adaptive time stepping
    """
    if external_torques is None:
        external_torques = {'Mx': 0.0, 'My': 0.0, 'Mz': 0.0}
    
    # Initialize arrays
    n_steps = int(duration / dt) + 1
    time = np.linspace(0, duration, n_steps)
    
    omega_x = np.zeros(n_steps)
    omega_y = np.zeros(n_steps)
    omega_z = np.zeros(n_steps)
    omega_dot_x = np.zeros(n_steps)
    omega_dot_y = np.zeros(n_steps)
    omega_dot_z = np.zeros(n_steps)
    kinetic_energy = np.zeros(n_steps)
    angular_momentum = np.zeros(n_steps)
    
    # Set initial conditions
    omega_x[0] = initial_rates['omega_x']
    omega_y[0] = initial_rates['omega_y']
    omega_z[0] = initial_rates['omega_z']
    
    # Propagate using production backend
    for i in range(n_steps):
        # Create input for production function
        attitude_input = AttitudeInput(
            inertia_x=inertia['Ix'],
            inertia_y=inertia['Iy'],
            inertia_z=inertia['Iz'],
            omega_x=omega_x[i],
            omega_y=omega_y[i],
            omega_z=omega_z[i],
            torque_x=external_torques['Mx'],
            torque_y=external_torques['My'],
            torque_z=external_torques['Mz'],
        )
        
        # Call production backend function
        result = compute_attitude_dynamics(attitude_input)
        
        # Store results
        omega_dot_x[i] = result.omega_dot_x
        omega_dot_y[i] = result.omega_dot_y
        omega_dot_z[i] = result.omega_dot_z
        kinetic_energy[i] = result.rotational_kinetic_energy
        angular_momentum[i] = result.angular_momentum_magnitude
        
        # Integrate to next step (Euler method)
        if i < n_steps - 1:
            omega_x[i + 1] = omega_x[i] + omega_dot_x[i] * dt
            omega_y[i + 1] = omega_y[i] + omega_dot_y[i] * dt
            omega_z[i + 1] = omega_z[i] + omega_dot_z[i] * dt
    
    return {
        'time': time,
        'omega_x': omega_x,
        'omega_y': omega_y,
        'omega_z': omega_z,
        'omega_dot_x': omega_dot_x,
        'omega_dot_y': omega_dot_y,
        'omega_dot_z': omega_dot_z,
        'kinetic_energy': kinetic_energy,
        'angular_momentum': angular_momentum,
        'assumptions': result.assumptions,
    }

In [ ]:
# ============================================================================
# Run Propagation: Torque-Free Motion
# ============================================================================
# Propagate for 60 seconds with no external torques

DURATION = 60.0  # seconds
DT = 0.01        # 10 ms time step

print(f"Propagating attitude for {DURATION} seconds (dt = {DT*1000:.0f} ms)...")

results = propagate_attitude(
    inertia=INERTIA,
    initial_rates=INITIAL_RATES,
    duration=DURATION,
    dt=DT
)

print(f"\nPropagation complete!")
print(f"  Total steps: {len(results['time'])}")
print(f"\nBackend assumptions used:")
for assumption in results['assumptions']:
    print(f"  - {assumption}")

---

## Results Visualization <a id='results-visualization'></a>

In [ ]:
# ============================================================================
# Plot 1: Angular Rates vs Time
# ============================================================================

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Convert to deg/s for plotting
omega_x_deg = np.degrees(results['omega_x'])
omega_y_deg = np.degrees(results['omega_y'])
omega_z_deg = np.degrees(results['omega_z'])

axes[0].plot(results['time'], omega_x_deg, 'b-', label=r'$\omega_x$')
axes[0].set_ylabel(r'$\omega_x$ [°/s]')
axes[0].set_title('Angular Rates vs Time (Torque-Free Motion)')
axes[0].legend(loc='upper right')

axes[1].plot(results['time'], omega_y_deg, 'g-', label=r'$\omega_y$')
axes[1].set_ylabel(r'$\omega_y$ [°/s]')
axes[1].legend(loc='upper right')

axes[2].plot(results['time'], omega_z_deg, 'r-', label=r'$\omega_z$')
axes[2].set_ylabel(r'$\omega_z$ [°/s]')
axes[2].set_xlabel('Time [s]')
axes[2].legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# Plot 2: All Angular Rates on Same Axes
# ============================================================================

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(results['time'], omega_x_deg, 'b-', label=r'$\omega_x$ (minor axis)')
ax.plot(results['time'], omega_y_deg, 'g-', label=r'$\omega_y$ (intermediate axis)')
ax.plot(results['time'], omega_z_deg, 'r-', label=r'$\omega_z$ (major axis)')

ax.set_xlabel('Time [s]')
ax.set_ylabel('Angular Rate [°/s]')
ax.set_title('Angular Rates vs Time - Combined View')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# Plot 3: Conservation Laws - Energy and Angular Momentum
# ============================================================================
# For torque-free motion, both should be conserved

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Kinetic Energy
axes[0].plot(results['time'], results['kinetic_energy'], 'b-', linewidth=2)
axes[0].set_ylabel('Kinetic Energy [J]')
axes[0].set_title('Conservation Laws Verification')

# Add reference line for initial value
T0 = results['kinetic_energy'][0]
axes[0].axhline(y=T0, color='r', linestyle='--', alpha=0.7, label=f'Initial: {T0:.4f} J')
axes[0].legend()

# Calculate percent drift
T_drift = (results['kinetic_energy'][-1] - T0) / T0 * 100
axes[0].text(0.98, 0.05, f'Drift: {T_drift:.4f}%', transform=axes[0].transAxes, 
             ha='right', fontsize=10, color='red' if abs(T_drift) > 0.1 else 'green')

# Angular Momentum
axes[1].plot(results['time'], results['angular_momentum'], 'g-', linewidth=2)
axes[1].set_ylabel('Angular Momentum [kg·m²/s]')
axes[1].set_xlabel('Time [s]')

# Add reference line for initial value
H0 = results['angular_momentum'][0]
axes[1].axhline(y=H0, color='r', linestyle='--', alpha=0.7, label=f'Initial: {H0:.4f} kg·m²/s')
axes[1].legend()

# Calculate percent drift
H_drift = (results['angular_momentum'][-1] - H0) / H0 * 100
axes[1].text(0.98, 0.05, f'Drift: {H_drift:.4f}%', transform=axes[1].transAxes, 
             ha='right', fontsize=10, color='red' if abs(H_drift) > 0.1 else 'green')

plt.tight_layout()
plt.show()

print(f"\nConservation Analysis:")
print(f"  Kinetic Energy: {T0:.4f} J → {results['kinetic_energy'][-1]:.4f} J (drift: {T_drift:.4f}%)")
print(f"  Angular Momentum: {H0:.4f} kg·m²/s → {results['angular_momentum'][-1]:.4f} kg·m²/s (drift: {H_drift:.4f}%)")
print(f"\n  Note: Drift is expected with Euler integration.")
print(f"        Consider RK4 for better conservation.")

In [ ]:
# ============================================================================
# Plot 4: Phase Portrait (omega_x vs omega_y)
# ============================================================================
# This shows the coupled motion in the transverse plane

fig, ax = plt.subplots(figsize=(10, 10))

# Plot trajectory with color gradient for time
points = ax.scatter(omega_x_deg, omega_y_deg, c=results['time'], 
                    cmap='viridis', s=1, label='Trajectory')

# Mark start and end points
ax.plot(omega_x_deg[0], omega_y_deg[0], 'go', markersize=15, label='Start')
ax.plot(omega_x_deg[-1], omega_y_deg[-1], 'r^', markersize=15, label='End')

ax.set_xlabel(r'$\omega_x$ [°/s]')
ax.set_ylabel(r'$\omega_y$ [°/s]')
ax.set_title('Phase Portrait: Transverse Angular Rates')
ax.legend()
ax.set_aspect('equal')

# Add colorbar
cbar = plt.colorbar(points)
cbar.set_label('Time [s]')

plt.tight_layout()
plt.show()

---

## Validation Cases <a id='validation-cases'></a>

Run specific test cases to validate physical behavior.

In [ ]:
# ============================================================================
# Validation Case 1: Pure Spin About Major Axis (Should be Stable)
# ============================================================================

print("="*60)
print("VALIDATION CASE 1: Pure Spin About Major Axis (Z)")
print("="*60)
print("Expected: Stable rotation with small perturbations remaining bounded")
print()

major_axis_spin = {
    'omega_x': deg_to_rad(0.1),   # Small perturbation
    'omega_y': deg_to_rad(0.1),   # Small perturbation
    'omega_z': deg_to_rad(10.0),  # Primary spin
}

results_major = propagate_attitude(
    inertia=INERTIA,
    initial_rates=major_axis_spin,
    duration=120.0,
    dt=0.01
)

# Check if transverse rates stay bounded
max_omega_x = np.max(np.abs(np.degrees(results_major['omega_x'])))
max_omega_y = np.max(np.abs(np.degrees(results_major['omega_y'])))

print(f"Initial: ωx = 0.1°/s, ωy = 0.1°/s, ωz = 10.0°/s")
print(f"Max |ωx| over 120s: {max_omega_x:.2f}°/s")
print(f"Max |ωy| over 120s: {max_omega_y:.2f}°/s")
print(f"\nResult: {'STABLE (as expected)' if max_omega_x < 1.0 and max_omega_y < 1.0 else 'UNSTABLE (unexpected!)'}")

In [ ]:
# ============================================================================
# Validation Case 2: Pure Spin About Intermediate Axis (Should be Unstable)
# ============================================================================

print("="*60)
print("VALIDATION CASE 2: Pure Spin About Intermediate Axis (Y)")
print("="*60)
print("Expected: UNSTABLE - small perturbations should grow exponentially")
print("          This is the 'tennis racket theorem' or Dzhanibekov effect")
print()

intermediate_axis_spin = {
    'omega_x': deg_to_rad(0.01),  # Tiny perturbation
    'omega_y': deg_to_rad(10.0),  # Primary spin about intermediate axis
    'omega_z': deg_to_rad(0.01),  # Tiny perturbation
}

results_intermediate = propagate_attitude(
    inertia=INERTIA,
    initial_rates=intermediate_axis_spin,
    duration=120.0,
    dt=0.01
)

# Plot the instability
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(results_intermediate['time'], np.degrees(results_intermediate['omega_x']), 'b-')
axes[0].set_ylabel(r'$\omega_x$ [°/s]')
axes[0].set_title('Intermediate Axis Instability (Tennis Racket Theorem)')

axes[1].plot(results_intermediate['time'], np.degrees(results_intermediate['omega_y']), 'g-')
axes[1].set_ylabel(r'$\omega_y$ [°/s]')

axes[2].plot(results_intermediate['time'], np.degrees(results_intermediate['omega_z']), 'r-')
axes[2].set_ylabel(r'$\omega_z$ [°/s]')
axes[2].set_xlabel('Time [s]')

plt.tight_layout()
plt.show()

# Check for instability
max_omega_x = np.max(np.abs(np.degrees(results_intermediate['omega_x'])))
max_omega_z = np.max(np.abs(np.degrees(results_intermediate['omega_z'])))

print(f"\nInitial: ωx = 0.01°/s, ωy = 10.0°/s, ωz = 0.01°/s")
print(f"Max |ωx| over 120s: {max_omega_x:.2f}°/s")
print(f"Max |ωz| over 120s: {max_omega_z:.2f}°/s")
print(f"\nResult: {'UNSTABLE (as expected - perturbations grew!)' if max_omega_x > 1.0 or max_omega_z > 1.0 else 'Stable (unexpected - try longer duration)'}")

In [ ]:
# ============================================================================
# Validation Case 3: Applied Constant Torque
# ============================================================================

print("="*60)
print("VALIDATION CASE 3: Constant Torque Application")
print("="*60)
print("Expected: Linear increase in angular momentum about torque axis")
print()

zero_initial = {
    'omega_x': 0.0,
    'omega_y': 0.0,
    'omega_z': 0.0,
}

constant_torque = {
    'Mx': 0.0,
    'My': 0.0,
    'Mz': 1.0,  # 1 N·m about Z-axis
}

results_torque = propagate_attitude(
    inertia=INERTIA,
    initial_rates=zero_initial,
    external_torques=constant_torque,
    duration=30.0,
    dt=0.01
)

# For constant torque about Z: ω_z = (M_z / I_z) * t
expected_omega_z = (constant_torque['Mz'] / INERTIA['Iz']) * results_torque['time']

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(results_torque['time'], np.degrees(results_torque['omega_z']), 'b-', 
        linewidth=2, label='Simulated')
ax.plot(results_torque['time'], np.degrees(expected_omega_z), 'r--', 
        linewidth=2, label='Expected (analytical)')

ax.set_xlabel('Time [s]')
ax.set_ylabel(r'$\omega_z$ [°/s]')
ax.set_title(f'Response to Constant Torque: Mz = {constant_torque["Mz"]} N·m')
ax.legend()

plt.tight_layout()
plt.show()

# Check final value
expected_final = np.degrees((constant_torque['Mz'] / INERTIA['Iz']) * 30.0)
actual_final = np.degrees(results_torque['omega_z'][-1])
error_pct = abs(actual_final - expected_final) / expected_final * 100

print(f"After 30s with Mz = 1 N·m:")
print(f"  Expected ωz: {expected_final:.4f} °/s")
print(f"  Actual ωz:   {actual_final:.4f} °/s")
print(f"  Error: {error_pct:.4f}%")

---

## Future Fidelity Improvements

The following enhancements would improve the accuracy and applicability of the attitude dynamics model:

### Backend Service Improvements

```
TODO: Add support for non-principal axis inertia tensors
      - Accept full 3x3 inertia matrix
      - Use generalized Euler equations: I·ω̇ + ω × (I·ω) = M

TODO: Add quaternion kinematics for attitude propagation
      - Track orientation, not just rates
      - Avoid gimbal lock issues with Euler angles

TODO: Add gravity gradient torque model
      - M_gg = (3μ/r³) * r̂ × (I · r̂)
      - Important for nadir-pointing spacecraft

TODO: Add magnetic disturbance torque
      - Residual dipole interaction with Earth's field

TODO: Add solar radiation pressure torque
      - Asymmetric spacecraft geometry effects

TODO: Add atmospheric drag torque (LEO)
      - CP-CM offset effects
```

### Notebook/Integration Improvements

```
TODO: Implement RK4 integrator for better conservation
TODO: Add 3D visualization of spacecraft attitude
TODO: Add Monte Carlo capability for sensitivity analysis
TODO: Add comparison with analytical solutions (polhode)
TODO: Add reaction wheel/CMG dynamics
```

---

## Summary

This notebook demonstrated:

1. **Importing and using production backend code** for attitude dynamics
2. **Euler's equations** governing rigid-body rotation
3. **Numerical propagation** of angular rates over time
4. **Conservation laws** for torque-free motion
5. **Stability analysis** of principal axis rotations
6. **The tennis racket theorem** (intermediate axis instability)

Key findings:
- The backend correctly implements Euler's equations
- Conservation laws are approximately preserved (limited by Euler integration)
- Physical behavior matches theoretical predictions

For production use, consider upgrading to RK4 integration and adding attitude representation (quaternions or DCM).